In [1]:
"""
Cross-Attention Representational Capacity Experiments (v2)
Fixed: consistent positional encodings, output MLPs, cleaner data generation.

Key changes from v1:
- Fixed positional encodings (shared across samples, not random per sample)
- Output MLP after attention for all models
- Simpler, more direct encoding of selection sets
- Proper evaluation logging

Usage: python cross_sa_experiments_v2.py
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


# ============================================================
# Data Generation
# ============================================================

def make_positional_encoding(N, d_pos):
    """Fixed sinusoidal positional encoding, shared across all samples."""
    pos = torch.arange(N).float().unsqueeze(1)  # (N, 1)
    div = torch.exp(torch.arange(0, d_pos, 2).float() * (-np.log(10000.0) / d_pos))
    pe = torch.zeros(N, d_pos)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div[:d_pos // 2])
    return pe


def generate_cross_sa_data(N, q, d_prime, n_samples, pe_source, pe_target):
    """
    Generate CrossSA instances with fixed positional encodings.

    Source input x_i = [pe_source[i]; pe_target[S(i,0)]; ...; pe_target[S(i,q-1)]]
    Target input y_j = [w_j; pe_target[j]]
    Label = (1/q) sum_{j in S(i)} w_j
    """
    d_pos = pe_source.shape[1]
    d_x = d_pos * (1 + q)  # own position + q target positions
    d_y = d_prime + d_pos   # data + position

    all_X = []
    all_Y = []
    all_S = []
    all_T = []

    for _ in range(n_samples):
        # Random data vectors on unit sphere
        W = torch.randn(N, d_prime)
        W = W / W.norm(dim=1, keepdim=True).clamp(min=1e-8)

        # Random selection sets
        S = torch.zeros(N, q, dtype=torch.long)
        for i in range(N):
            S[i] = torch.tensor(random.sample(range(N), q))

        # Targets
        T = torch.stack([W[S[i]].mean(dim=0) for i in range(N)])

        # Source input: own PE + PE of each selected target
        X = torch.zeros(N, d_x)
        for i in range(N):
            X[i, :d_pos] = pe_source[i]
            for k in range(q):
                X[i, d_pos + k * d_pos: d_pos + (k + 1) * d_pos] = pe_target[S[i, k]]

        # Target input: data vector + PE
        Y = torch.cat([W, pe_target], dim=1)

        all_X.append(X)
        all_Y.append(Y)
        all_S.append(S)
        all_T.append(T)

    return torch.stack(all_X), torch.stack(all_Y), torch.stack(all_S), torch.stack(all_T)


# ============================================================
# Models
# ============================================================

class CrossAttentionModel(nn.Module):
    """Cross-attention with input MLPs and output MLP."""

    def __init__(self, d_x, d_y, d_out, m, hidden=64):
        super().__init__()
        self.m = m
        self.query_mlp = nn.Sequential(nn.Linear(d_x, hidden), nn.ReLU(), nn.Linear(hidden, m))
        self.key_mlp = nn.Sequential(nn.Linear(d_y, hidden), nn.ReLU(), nn.Linear(hidden, m))
        self.value_mlp = nn.Sequential(nn.Linear(d_y, hidden), nn.ReLU(), nn.Linear(hidden, m))
        self.output_mlp = nn.Sequential(nn.Linear(m, hidden), nn.ReLU(), nn.Linear(hidden, d_out))

    def forward(self, X, Y, return_attn=False):
        Q = self.query_mlp(X)
        K = self.key_mlp(Y)
        V = self.value_mlp(Y)

        scores = torch.bmm(Q, K.transpose(1, 2)) / np.sqrt(self.m)
        attn = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn, V)
        output = self.output_mlp(context)

        if return_attn:
            return output, attn
        return output


class SelfAttentionConcatModel(nn.Module):
    """Self-attention on [X; Y] concat with input and output MLPs."""

    def __init__(self, d_x, d_y, d_out, m, hidden=64):
        super().__init__()
        self.m = m
        d_max = max(d_x, d_y) + 1  # +1 for stream indicator
        self.embed_x = nn.Sequential(nn.Linear(d_x + 1, hidden), nn.ReLU(), nn.Linear(hidden, m))
        self.embed_y = nn.Sequential(nn.Linear(d_y + 1, hidden), nn.ReLU(), nn.Linear(hidden, m))
        self.Q = nn.Linear(m, m)
        self.K = nn.Linear(m, m)
        self.V = nn.Linear(m, m)
        self.output_mlp = nn.Sequential(nn.Linear(m, hidden), nn.ReLU(), nn.Linear(hidden, d_out))

    def forward(self, X, Y, return_attn=False):
        B, Nx, _ = X.shape
        Ny = Y.shape[1]

        # Add stream indicator
        x_ind = torch.cat([X, torch.zeros(B, Nx, 1)], dim=-1)
        y_ind = torch.cat([Y, torch.ones(B, Ny, 1)], dim=-1)

        X_emb = self.embed_x(x_ind)
        Y_emb = self.embed_y(y_ind)
        Z = torch.cat([X_emb, Y_emb], dim=1)

        Q = self.Q(Z)
        K = self.K(Z)
        V = self.V(Z)

        scores = torch.bmm(Q, K.transpose(1, 2)) / np.sqrt(self.m)
        attn = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn, V)

        output = self.output_mlp(context[:, :Nx, :])

        if return_attn:
            return output, attn
        return output


class MLPModel(nn.Module):
    """MLP baseline on flattened input."""

    def __init__(self, N, d_x, d_y, d_out, hidden=512):
        super().__init__()
        self.N = N
        self.d_out = d_out
        self.net = nn.Sequential(
            nn.Linear(N * d_x + N * d_y, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, N * d_out),
        )

    def forward(self, X, Y, return_attn=False):
        B = X.shape[0]
        flat = torch.cat([X.reshape(B, -1), Y.reshape(B, -1)], dim=1)
        out = self.net(flat).reshape(B, self.N, self.d_out)
        if return_attn:
            return out, None
        return out


# ============================================================
# Training utilities
# ============================================================

def train_model(model, X_train, Y_train, T_train, X_test, Y_test, T_test,
                epochs=500, batch_size=32, lr=1e-3):
    """Train and return per-epoch train/test losses."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    train_losses = []
    test_losses = []
    n_train = X_train.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0
        n_batches = 0

        for start in range(0, n_train, batch_size):
            idx = perm[start:start + batch_size]
            pred = model(X_train[idx], Y_train[idx])
            loss = ((pred - T_train[idx]) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()

        # Evaluate on full sets
        model.eval()
        with torch.no_grad():
            train_pred = model(X_train, Y_train)
            train_mse = ((train_pred - T_train) ** 2).mean().item()

            test_pred = model(X_test, Y_test)
            test_mse = ((test_pred - T_test) ** 2).mean().item()

        train_losses.append(train_mse)
        test_losses.append(test_mse)

        if (epoch + 1) % 100 == 0:
            print(f"    Epoch {epoch+1}: train={train_mse:.6f}, test={test_mse:.6f}")

    return train_losses, test_losses


# ============================================================
# Experiment 1: Learning Curves
# ============================================================

def experiment_learning_curves(N=15, q=3, d_prime=4, m=64, epochs=500):
    print(f"Config: N={N}, q={q}, d'={d_prime}, m={m}, epochs={epochs}")

    d_pos = max(4, int(np.ceil(2 + 2 * np.log(N))))
    pe_s = make_positional_encoding(N, d_pos)
    pe_t = make_positional_encoding(N, d_pos)
    d_x = d_pos * (1 + q)
    d_y = d_prime + d_pos

    print("Generating data...")
    X_tr, Y_tr, S_tr, T_tr = generate_cross_sa_data(N, q, d_prime, 1000, pe_s, pe_t)
    X_te, Y_te, S_te, T_te = generate_cross_sa_data(N, q, d_prime, 200, pe_s, pe_t)

    results = {}

    configs = [
        ("Cross-Attention", CrossAttentionModel(d_x, d_y, d_prime, m)),
        ("Self-Attention (concat)", SelfAttentionConcatModel(d_x, d_y, d_prime, m)),
        ("MLP", MLPModel(N, d_x, d_y, d_prime, hidden=512)),
    ]

    for name, model in configs:
        n_params = sum(p.numel() for p in model.parameters())
        print(f"\n  Training {name} ({n_params:,} params)...")
        tr, te = train_model(model, X_tr, Y_tr, T_tr, X_te, Y_te, T_te,
                             epochs=epochs, lr=1e-3)
        results[name] = {"train": tr, "test": te}

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {"Cross-Attention": "blue", "Self-Attention (concat)": "red", "MLP": "green"}

    for name in results:
        ax.plot(results[name]["train"], color=colors[name], linestyle="--",
                alpha=0.5, label=f"{name} (train)")
        ax.plot(results[name]["test"], color=colors[name],
                label=f"{name} (test)")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.set_title(f"CrossSA Learning Curves (N={N}, q={q})")
    ax.legend(fontsize=8)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    fig.savefig("fig1_learning_curves.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("\nSaved fig1_learning_curves.png")

    return results


# ============================================================
# Experiment 2: Attention Matrix Convergence
# ============================================================

def experiment_attn_convergence(N=15, q=3, d_prime=4, m=64, epochs=2000):
    print(f"Config: N={N}, q={q}, m={m}, epochs={epochs}")

    d_pos = max(4, int(np.ceil(2 + 2 * np.log(N))))
    pe_s = make_positional_encoding(N, d_pos)
    pe_t = make_positional_encoding(N, d_pos)
    d_x = d_pos * (1 + q)
    d_y = d_prime + d_pos

    X_tr, Y_tr, S_tr, T_tr = generate_cross_sa_data(N, q, d_prime, 1000, pe_s, pe_t)

    model = CrossAttentionModel(d_x, d_y, d_prime, m)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Track one example
    x_ex = X_tr[0:1]
    y_ex = Y_tr[0:1]
    s_ex = S_tr[0]

    # Ground truth attention
    gt = torch.zeros(N, N)
    for i in range(N):
        for k in range(q):
            gt[i, s_ex[i, k]] = 1.0 / q

    checkpoints = [0, 200, 500, 2000]
    attn_snapshots = {}

    batch_size = 32
    for epoch in range(epochs + 1):
        if epoch in checkpoints:
            model.eval()
            with torch.no_grad():
                _, attn = model(x_ex, y_ex, return_attn=True)
            attn_snapshots[epoch] = attn[0].numpy()

        if epoch < epochs:
            model.train()
            idx = torch.randperm(1000)[:batch_size]
            pred = model(X_tr[idx], Y_tr[idx])
            loss = ((pred - T_tr[idx]) ** 2).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 500 == 0:
            model.eval()
            with torch.no_grad():
                pred = model(X_tr[:200], Y_tr[:200])
                mse = ((pred - T_tr[:200]) ** 2).mean().item()
            print(f"    Epoch {epoch+1}: MSE={mse:.6f}")

    # Plot
    n_plots = len(checkpoints) + 1
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 3.5))

    axes[0].imshow(gt.numpy(), cmap="Blues", vmin=0, vmax=0.4)
    axes[0].set_title("Ground Truth", fontsize=10)
    axes[0].set_ylabel("Source i")

    for idx, cp in enumerate(checkpoints):
        ax = axes[idx + 1]
        ax.imshow(attn_snapshots[cp], cmap="Blues", vmin=0, vmax=0.4)
        ax.set_title(f"T = {cp}", fontsize=10)

        # Mark ground truth positions
        for i in range(N):
            for k in range(q):
                rect = plt.Rectangle((s_ex[i, k] - 0.5, i - 0.5), 1, 1,
                                     fill=False, edgecolor="red", linewidth=0.7)
                ax.add_patch(rect)

    for ax in axes:
        ax.set_xlabel("Target j", fontsize=9)

    plt.tight_layout()
    fig.savefig("fig2_attn_convergence.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved fig2_attn_convergence.png")


# ============================================================
# Experiment 3: Embedding Dimension Scaling
# ============================================================

def experiment_embedding_scaling(N=15, q=3, d_prime=4, epochs=400):
    print(f"Config: N={N}, q={q}")

    d_pos = max(4, int(np.ceil(2 + 2 * np.log(N))))
    pe_s = make_positional_encoding(N, d_pos)
    pe_t = make_positional_encoding(N, d_pos)
    d_x = d_pos * (1 + q)
    d_y = d_prime + d_pos

    X_tr, Y_tr, _, T_tr = generate_cross_sa_data(N, q, d_prime, 1000, pe_s, pe_t)
    X_te, Y_te, _, T_te = generate_cross_sa_data(N, q, d_prime, 200, pe_s, pe_t)

    m_values = [4, 8, 16, 32, 48, 64, 96, 128]
    m_threshold = q * np.log(N)

    results_cross = []
    results_self = []

    for m in m_values:
        for model_type, result_list in [("cross", results_cross), ("self", results_self)]:
            if model_type == "cross":
                model = CrossAttentionModel(d_x, d_y, d_prime, m)
            else:
                model = SelfAttentionConcatModel(d_x, d_y, d_prime, m)

            _, test_losses = train_model(model, X_tr, Y_tr, T_tr,
                                         X_te, Y_te, T_te,
                                         epochs=epochs, lr=1e-3)
            final_mse = min(test_losses)  # best test MSE
            result_list.append(final_mse)
            print(f"  m={m:3d}, {model_type:5s}: best test MSE = {final_mse:.6f}")

    # Plot
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(m_values, results_cross, "bo-", linewidth=2, markersize=6, label="Cross-Attention")
    ax.plot(m_values, results_self, "rs-", linewidth=2, markersize=6, label="Self-Attention (concat)")
    ax.axvline(x=m_threshold, color="gray", linestyle="--", alpha=0.7,
               label=f"m = q ln N = {m_threshold:.1f}")
    ax.set_xlabel("Embedding dimension m")
    ax.set_ylabel("Best Test MSE")
    ax.set_title(f"Embedding Scaling (N={N}, q={q})")
    ax.legend()
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    fig.savefig("fig3_embedding_scaling.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved fig3_embedding_scaling.png")


# ============================================================
# Experiment 4: Attention Leakage
# ============================================================

def experiment_attention_leakage(N=15, q=3, d_prime=4, m=64, epochs=500):
    print(f"Config: N={N}, q={q}, m={m}")

    d_pos = max(4, int(np.ceil(2 + 2 * np.log(N))))
    pe_s = make_positional_encoding(N, d_pos)
    pe_t = make_positional_encoding(N, d_pos)
    d_x = d_pos * (1 + q)
    d_y = d_prime + d_pos

    X_tr, Y_tr, _, T_tr = generate_cross_sa_data(N, q, d_prime, 1000, pe_s, pe_t)

    model = SelfAttentionConcatModel(d_x, d_y, d_prime, m)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    batch_size = 32

    leakage = []

    for epoch in range(epochs):
        model.train()
        idx = torch.randperm(1000)[:batch_size]
        pred = model(X_tr[idx], Y_tr[idx])
        loss = ((pred - T_tr[idx]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                _, attn = model(X_tr[:50], Y_tr[:50], return_attn=True)
                # Source rows attending to X vs Y
                src_attn = attn[:, :N, :]
                mass_xx = src_attn[:, :, :N].sum(dim=-1).mean().item()
                mass_xy = src_attn[:, :, N:].sum(dim=-1).mean().item()
            leakage.append({"epoch": epoch + 1, "waste": mass_xx, "useful": mass_xy})

    # Plot leakage over time
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    epochs_list = [l["epoch"] for l in leakage]
    ax1.plot(epochs_list, [l["useful"] for l in leakage], "b-", linewidth=2,
             label="A_XY (useful: cross-stream)")
    ax1.plot(epochs_list, [l["waste"] for l in leakage], "r-", linewidth=2,
             label="A_XX (waste: within-stream)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Attention Mass (source rows)")
    ax1.set_title("Attention Mass Distribution Over Training")
    ax1.legend()
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)

    # Plot final attention matrix
    model.eval()
    with torch.no_grad():
        _, attn = model(X_tr[:1], Y_tr[:1], return_attn=True)

    attn_np = attn[0].numpy()
    im = ax2.imshow(attn_np, cmap="Blues", vmin=0)
    ax2.axhline(y=N - 0.5, color="red", linewidth=1.5, linestyle="--")
    ax2.axvline(x=N - 0.5, color="red", linewidth=1.5, linestyle="--")
    ax2.set_title("Final Self-Attention Matrix [X | Y]")
    ax2.set_xlabel("Token index")
    ax2.set_ylabel("Token index")
    ax2.text(N // 2, N // 2, "A_XX\n(waste)", ha="center", va="center",
             color="red", fontsize=9, fontweight="bold")
    ax2.text(N + N // 2, N // 2, "A_XY\n(useful)", ha="center", va="center",
             color="blue", fontsize=9, fontweight="bold")
    ax2.text(N // 2, N + N // 2, "A_YX", ha="center", va="center",
             color="gray", fontsize=9)
    ax2.text(N + N // 2, N + N // 2, "A_YY", ha="center", va="center",
             color="gray", fontsize=9)
    plt.colorbar(im, ax=ax2)

    plt.tight_layout()
    fig.savefig("fig4_attention_leakage.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved fig4_attention_leakage.png")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Experiment 1: Learning Curves")
    print("=" * 60)
    experiment_learning_curves()

    print("\n" + "=" * 60)
    print("Experiment 2: Attention Matrix Convergence")
    print("=" * 60)
    experiment_attn_convergence()

    print("\n" + "=" * 60)
    print("Experiment 3: Embedding Dimension Scaling")
    print("=" * 60)
    experiment_embedding_scaling()

    print("\n" + "=" * 60)
    print("Experiment 4: Attention Leakage")
    print("=" * 60)
    experiment_attention_leakage()

    print("\n" + "=" * 60)
    print("ALL EXPERIMENTS COMPLETE")
    print("Outputs: fig1_learning_curves.png")
    print("         fig2_attn_convergence.png")
    print("         fig3_embedding_scaling.png")
    print("         fig4_attention_leakage.png")
    print("=" * 60)

Experiment 1: Learning Curves
Config: N=15, q=3, d'=4, m=64, epochs=500
Generating data...

  Training Cross-Attention (20,676 params)...
    Epoch 100: train=0.037226, test=0.039873
    Epoch 200: train=0.020092, test=0.022247
    Epoch 300: train=0.017343, test=0.019920
    Epoch 400: train=0.016687, test=0.019454
    Epoch 500: train=0.016567, test=0.019356

  Training Self-Attention (concat) (28,292 params)...
    Epoch 100: train=0.041783, test=0.044986
    Epoch 200: train=0.035549, test=0.039859
    Epoch 300: train=0.028774, test=0.032895
    Epoch 400: train=0.027375, test=0.031676
    Epoch 500: train=0.027097, test=0.031474

  Training MLP (631,868 params)...
    Epoch 100: train=0.007962, test=0.141538
    Epoch 200: train=0.003443, test=0.169590
    Epoch 300: train=0.002056, test=0.183671
    Epoch 400: train=0.001452, test=0.190506
    Epoch 500: train=0.001355, test=0.191752

Saved fig1_learning_curves.png

Experiment 2: Attention Matrix Convergence
Config: N=15, q=3, m